# kaggle-llm-server — Run All

**Перед запуском:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**.

Этот ноутбук: 1) распаковывает проект, 2) проверяет железо, 3) ставит зависимости,
4) собирает llama.cpp с CUDA, 5) качает модель, 6) подбирает оптимальные параметры,
7) запускает OpenAI-совместимый сервер + Web UI, 8) публикует их через туннель.

Просто нажмите **Run All**. В конце в выводе появятся публичные ссылки.

In [ ]:
!nvidia-smi

## 1. Получение исходников проекта из GitHub

Ноутбук при каждом запуске получает актуальную версию из репозитория **PoFigiHubIO/KeglaAI**.
Репозиторий публичный, поэтому токен GitHub в Kaggle не требуется. Исходники копируются в
`/kaggle/working`, а модели и готовая сборка сохраняются между перезапусками сессии.


In [ ]:
import os, shutil, subprocess

REPO_URL = "https://github.com/PoFigiHubIO/KeglaAI.git"
BRANCH = "main"
PROJECT_SUBDIR = "kaggle-llm-server"
REPO_DIR = "/kaggle/working/KeglaAI"
WORK_DIR = f"{REPO_DIR}/{PROJECT_SUBDIR}"

# Клонируем при первом запуске; при повторном — получаем свежую версию кода.
# Не используем git clean: models/, llama.cpp/ и logs/ остаются в Persistence.
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH, "--depth", "1"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)

assert os.path.isfile(os.path.join(WORK_DIR, "start.py")), f"Проект не найден: {WORK_DIR}"
os.chdir(WORK_DIR)
print("Рабочая директория:", os.getcwd())
print("Версия исходников:", subprocess.check_output(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], text=True).strip())


## 2. (Опционально) Настройка модели и секретов

Отредактируйте `config.yaml` при необходимости, либо переопределите поля прямо здесь
перед запуском `start.py`. Для приватных репозиториев HuggingFace добавьте
`HF_TOKEN` через **Add-ons → Secrets** в Kaggle и раскомментируйте код ниже.

In [ ]:
# from kaggle_secrets import UserSecretsClient
# os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

# Пример: сменить модель прямо из ноутбука без ручного редактирования файла
# import yaml
# with open('config.yaml') as f: cfg = yaml.safe_load(f)
# cfg['model']['repo_id'] = 'bartowski/Meta-Llama-3.1-8B-Instruct-GGUF'
# cfg['model']['filename'] = 'Meta-Llama-3.1-8B-Instruct-Q5_K_M.gguf'
# with open('config.yaml', 'w') as f: yaml.safe_dump(cfg, f, allow_unicode=True)
print("Конфигурация готова (см. config.yaml)")

## 3. Полный запуск: hardware check → install → build → download → optimize → server → tunnel

In [ ]:
!python start.py

## 4. Проверка API

In [ ]:
import json

with open('logs/optimized_params.json') as f:
    print(json.dumps(json.load(f), indent=2, ensure_ascii=False))

!curl -s http://127.0.0.1:8080/v1/models

## 5. Keep-alive

Эта ячейка держит сессию Kaggle активной, пока сервер работает.
Остановите ячейку (interrupt), когда закончите пользоваться сервером.

In [ ]:
import time
print("Сервер запущен и доступен по публичной ссылке, показанной выше.")
print("Оставьте эту ячейку выполняться, чтобы сессия не завершилась.")
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("Остановлено пользователем.")